In [1]:
import requests

dataset_slug = "bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2025"
meta_url = f"https://www.data.gouv.fr/api/1/datasets/{dataset_slug}/"

dataset = requests.get(meta_url).json()

# Print all 2025 CSV files and their Resource IDs
for res in dataset["resources"]:
    if res["format"] == "csv" and "2025" in res["title"]:
        print(f"{res['title']} -> rid: {res['id']}")

vehicules-immatricule-baac-2025.csv -> rid: 9827fd15-eebc-461e-a4af-b48594ce3c6b
Vehicules_2025.csv -> rid: 58ba05bf-661d-4c59-b504-3d77336ccb8f
Usagers_2025.csv -> rid: 513b5a4a-20cc-4ee2-a0c8-5e5a82ad69a4
Lieux_2025.csv -> rid: 29db11b7-dad2-480f-943d-725ffdeb9a27
Caract_2025.csv -> rid: e5a1c21d-17cb-4ee0-b85f-8e34aba4d309


In [ ]:
import requests
import pandas as pd

dataset_slug = "bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2025"
meta_url = f"https://www.data.gouv.fr/api/1/datasets/{dataset_slug}/"
dataset = requests.get(meta_url).json()

# 1. Collect the 2025 CSV resources
resources = {
    res["title"]: res
    for res in dataset["resources"]
    if res["format"] == "csv" and "2025" in res["title"]
}
for title, res in resources.items():
    print(f"{title} -> rid: {res['id']}")


def load_resource(res, **kwargs):
    url = f"https://www.data.gouv.fr/fr/datasets/r/{res['id']}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()

    # Detect separator from the header line
    header = r.content[:2000].decode("utf-8-sig", errors="replace").splitlines()[0]
    sep = ";" if header.count(";") > header.count(",") else ","

    return pd.read_csv(
        io.BytesIO(r.content),
        sep=sep,
        encoding="utf-8-sig",   # also strips a BOM if present
        dtype={"Num_Acc": str, "id_vehicule": str, "num_veh": str},
        low_memory=False,
        **kwargs,
    )

dfs = {title: load_resource(res) for title, res in resources.items()}

for title, df in dfs.items():
    print(f"\n=== {title} ===")
    print(df.shape)
    print(df.head(3))